# **Analyzing llama3.1-7b architecture**

This notebook is boring and possibly useless for more technical users, but I need it in order to understand how to extract hooks and activations at different points from the forward pass.

I decided to operate in a diverse notebook for both diversification of the subject, and because I wanted to operate while keeping the full model in CPU, avoiding excessive memory usage.

## **Importing the model**

In [1]:
from transformers import AutoModelForCausalLM

from torch.nn import Module
from torch.mps import current_allocated_memory, recommended_max_memory

def allocated_memory(percentage=True) -> float:
    mem = current_allocated_memory()/recommended_max_memory()
    if percentage:
        mem /= 100
    return mem
##

print('SESSION INITIALIZED')
print(f'MPS availability for this session (recommended): {recommended_max_memory()}')
print(f'Currently allocated memory {allocated_memory()}%')

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
print(f'\nImporting model: {MODEL_NAME}')
llama_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path= MODEL_NAME,
    device_map = 'cpu',
)
print(f'MPS usage after importing: {allocated_memory()}%')

/Users/simone/miniconda3/envs/llm-steering/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SESSION INITIALIZED
MPS availability for this session (recommended): 26800603136
Currently allocated memory 0.0%

Importing model: meta-llama/Llama-3.1-8B-Instruct


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 4260.05it/s]


MPS usage after importing: 0.0%


In [21]:
print(llama_model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

## **Analyzing inheritance**

In [15]:
from transformers.models.llama.modeling_llama import LlamaForCausalLM
from itertools import chain

print('Summary of llama_model:')
print(f'type: {type(llama_model)}')

print()
print('Full tree of inheritance:')

obj = [LlamaForCausalLM]; i=0
while True:
    print(f'inheritance level {i}: {obj}')
    obj = set(chain.from_iterable([j.__bases__ for j in obj]))
    i+=1

    if obj == set([object]):
        break
    
# print(f'inherits from: {type(llama_model).__bases__}')


Summary of llama_model:
type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>

Full tree of inheritance:
inheritance level 0: [<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>]
inheritance level 1: {<class 'transformers.generation.utils.GenerationMixin'>, <class 'transformers.models.llama.modeling_llama.LlamaPreTrainedModel'>}
inheritance level 2: {<class 'transformers.generation.continuous_batching.continuous_api.ContinuousMixin'>, <class 'transformers.modeling_utils.PreTrainedModel'>}
inheritance level 3: {<class 'transformers.modeling_utils.ModuleUtilsMixin'>, <class 'transformers.integrations.peft.PeftAdapterMixin'>, <class 'torch.nn.modules.module.Module'>, <class 'transformers.utils.hub.PushToHubMixin'>, <class 'object'>, <class 'transformers.modeling_utils.EmbeddingAccessMixin'>}


We can see inheritance from various `transformers` metaclasses, but the real backbone is clearly the `torch.nn.modules.Module`, that possibly is the direct parent of `transformers.modeling_utils.PreTrainedModel`.

## **Identifying and accessing submodules**

In [25]:
list(llama_model.children())[0] # this is only the main model (without the 'head' linear decoding)

LlamaModel(
  (embed_tokens): Embedding(128256, 4096)
  (layers): ModuleList(
    (0-31): 32 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((4096,), eps=1e-05)
  (rotary_emb): LlamaRotaryEmbedding()
)

In [31]:
for section, params in llama_model.named_parameters():
    print(f'{section}, shape: {params.shape}')
##

model.embed_tokens.weight, shape: torch.Size([128256, 4096])
model.layers.0.self_attn.q_proj.weight, shape: torch.Size([4096, 4096])
model.layers.0.self_attn.k_proj.weight, shape: torch.Size([1024, 4096])
model.layers.0.self_attn.v_proj.weight, shape: torch.Size([1024, 4096])
model.layers.0.self_attn.o_proj.weight, shape: torch.Size([4096, 4096])
model.layers.0.mlp.gate_proj.weight, shape: torch.Size([14336, 4096])
model.layers.0.mlp.up_proj.weight, shape: torch.Size([14336, 4096])
model.layers.0.mlp.down_proj.weight, shape: torch.Size([4096, 14336])
model.layers.0.input_layernorm.weight, shape: torch.Size([4096])
model.layers.0.post_attention_layernorm.weight, shape: torch.Size([4096])
model.layers.1.self_attn.q_proj.weight, shape: torch.Size([4096, 4096])
model.layers.1.self_attn.k_proj.weight, shape: torch.Size([1024, 4096])
model.layers.1.self_attn.v_proj.weight, shape: torch.Size([1024, 4096])
model.layers.1.self_attn.o_proj.weight, shape: torch.Size([4096, 4096])
model.layers.1.m